# LoRA Research Notes Adapter

This notebook walks through the full pipeline: load the best adapter, run inference on a custom prompt, and evaluate the output against the format template.

**Requirements:** run from the repo root with the project's virtual environment active.
```bash
source .venv/bin/activate
jupyter notebook notebooks/demo.ipynb
```

## 1 · Setup

In [1]:
import sys, json, textwrap
from pathlib import Path

ROOT = Path().resolve().parent  # repo root when notebook is in notebooks/
sys.path.insert(0, str(ROOT / 'src'))

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

from eval_template import build_prompt, check_compliance, detect_device
from eval_rubric   import rubric_score
from constants     import SYSTEM_PROMPT

print('torch:', torch.__version__)
device = detect_device()
print('device:', device)

/Users/jeffery/Desktop/research_ass/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.11.0
device: mps


## 2 · Pick an adapter

By default we use the best experiment (`rank_64`). Change `ADAPTER_NAME` to any experiment in `outputs/experiments/`.

In [2]:
ADAPTER_NAME = 'rank_64'   # change to: baseline / rank_8 / rank_32 / epochs_5 / lr_1e-4 / lr_5e-4
ADAPTER_DIR  = ROOT / 'outputs' / 'experiments' / ADAPTER_NAME

meta = json.loads((ADAPTER_DIR / 'training_meta.json').read_text())
MODEL_ID = meta['model_id']
print(f'Base model : {MODEL_ID}')
print(f'Adapter    : {ADAPTER_NAME}  (r={meta["lora_r"]}, lr={meta["learning_rate"]}, epochs={meta["epochs"]})')

Base model : Qwen/Qwen2.5-1.5B-Instruct
Adapter    : rank_64  (r=64, lr=0.0002, epochs=3)


## 3 · Load base model + adapter

In [3]:
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Loading base model...')
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    device_map={'': device},
    trust_remote_code=True,
)
base.eval()

print('Attaching LoRA adapter...')
model = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
model.eval()
print('Ready.')

Loading tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading base model...
Attaching LoRA adapter...
Ready.


## 4 · Inference helper

In [4]:
def generate(prompt_text: str, max_new_tokens: int = 256) -> str:
    prompt = build_prompt(prompt_text, tokenizer)
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

print('generate() ready')

generate() ready


## 5 · Try your own prompt

In [5]:
MY_PROMPT = """
Mixture of Experts (MoE) is an architecture that activates only a sparse subset
of specialist sub-networks (experts) for each input token, allowing very large
total parameter counts while keeping per-token computation constant.
Models such as Mixtral 8x7B use a learned routing mechanism to select the
top-k experts per token at each layer.
""".strip()

output = generate(MY_PROMPT)
print(output)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Summary:
Mixture-of-experts architectures activate only a small fraction of expert subnetworks per token, enabling extremely deep models with nearly linear scaling.

Key Points:
- MoE reduces model size by many orders of magnitude compared to pure transformer stacks.
- Each expert specializes on a different part of the input and communicates via attention.
- The number of experts can be dynamically adjusted during training without retraining from scratch.

Limitation:
The routing mechanism must learn both which experts to activate and how much weight to assign to each one, making it challenging to train effectively.

Follow-up Question:
Does MoE's ability to scale to extremely long sequences come at the cost of significant performance degradation?


## 6 · Evaluate the output

In [6]:
compliance = check_compliance(output)
rubric     = rubric_score(output)

print('=== Format compliance ===')
for k, v in compliance.items():
    mark = '✓' if v else '✗'
    print(f'  {mark}  {k}')

print(f'\n=== Rubric score: {rubric["total"]} / 12 ===')
for dim in ('summary', 'bullets', 'limitation', 'followup'):
    print(f'  {dim:<12} {rubric[dim]} / 3')

=== Format compliance ===
  ✓  summary
  ✓  key_points
  ✓  limitation
  ✓  follow_up
  ✓  three_bullets
  ✓  bullet_count
  ✓  all_pass

=== Rubric score: 11 / 12 ===
  summary      3 / 3
  bullets      2 / 3
  limitation   3 / 3
  followup     3 / 3


## 7 · Side-by-side: base vs LoRA on the held-out test set

Run this cell to see the 10 held-out prompts evaluated against both the raw base model and the LoRA adapter.

In [7]:
TEST_PATH = ROOT / 'data' / 'test_prompts.jsonl'
prompts   = [json.loads(l) for l in TEST_PATH.read_text().splitlines() if l.strip()]

rows = []
for p in prompts:
    # Base (adapter disabled)
    model.disable_adapter_layers()
    base_out   = generate(p['input'])
    base_comp  = check_compliance(base_out)
    base_rub   = rubric_score(base_out)

    # LoRA adapter
    model.enable_adapter_layers()
    lora_out   = generate(p['input'])
    lora_comp  = check_compliance(lora_out)
    lora_rub   = rubric_score(lora_out)

    rows.append(dict(
        id=p['id'],
        base_pass=base_comp['all_pass'], base_rubric=base_rub['total'],
        lora_pass=lora_comp['all_pass'], lora_rubric=lora_rub['total'],
    ))
    status = '✓' if lora_comp['all_pass'] else '✗'
    print(f"{p['id']}  base={base_rub['total']:4.1f}/12  lora={lora_rub['total']:4.1f}/12  {status}")

base_rate = sum(r['base_pass'] for r in rows) / len(rows)
lora_rate = sum(r['lora_pass'] for r in rows) / len(rows)
print(f"\nCompliance  base={base_rate:.0%}  lora={lora_rate:.0%}")
print(f"Avg rubric  base={sum(r['base_rubric'] for r in rows)/len(rows):.2f}/12  "
      f"lora={sum(r['lora_rubric'] for r in rows)/len(rows):.2f}/12")

test_01  base= 4.0/12  lora= 7.0/12  ✓
test_02  base=11.0/12  lora=10.0/12  ✓
test_03  base= 9.0/12  lora=11.0/12  ✓
test_04  base=11.0/12  lora=11.0/12  ✓
test_05  base= 9.0/12  lora=11.0/12  ✓
test_06  base=10.0/12  lora= 9.0/12  ✓
test_07  base=10.0/12  lora=10.0/12  ✓
test_08  base=10.0/12  lora= 9.0/12  ✓
test_09  base=10.0/12  lora=10.0/12  ✓
test_10  base=12.0/12  lora=10.0/12  ✓
test_11  base=11.0/12  lora=12.0/12  ✓

Compliance  base=0%  lora=100%
Avg rubric  base=9.73/12  lora=10.00/12
